# VoiceHub fine-tuning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/training.ipynb)

Inspect support, validate a small authorized dataset, run exactly one optimizer step, then save and reload. Expensive stages are off by default.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("voicehub") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "voicehub[training] @ git+https://github.com/kadirnar/voicehub.git@main",
    ])

## 1. Configure

Keep `MAX_STEPS=1` until the loss is finite, intended parameters receive gradients, and save/reload succeeds.

In [ ]:
from pathlib import Path

RUN_TRAINING = False
RUN_RELOAD = False

MODEL_TYPE = "dia"
BASE_MODEL = "nari-labs/Dia-1.6B-0626"
DEVICE = "cuda"
MAX_STEPS = 1
OUTPUT_DIR = Path("artifacts/dia-smoke")
MANIFEST_PATH = Path("data/dia/manifest.jsonl")

## 2. Inspect support before loading weights

In [ ]:
from voicehub.training.arguments import TrainingArguments
from voicehub.training import get_training_spec, get_tts_dataset_spec

training_spec = get_training_spec(MODEL_TYPE)
dataset_spec = get_tts_dataset_spec(MODEL_TYPE)
print(training_spec.support.value, training_spec.family_name)
print(dataset_spec.architecture.value, dataset_spec.readiness.value)

smoke_arguments = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    logging_steps=1,
    save_steps=1,
    use_cpu=DEVICE == "cpu",
    report_to="none",
    seed=42,
    data_seed=42,
)

## 3. Load real data

The manifest must contain authorized audio, exact transcripts, stable IDs, and speaker or session groups. The model adapter owns final token, feature, or codec preparation.

In [ ]:
if RUN_TRAINING:
    from voicehub.training import TTSDataset

    if not MANIFEST_PATH.is_file():
        raise FileNotFoundError(MANIFEST_PATH)
    source_dataset = TTSDataset.from_manifest(
        MANIFEST_PATH,
        model_type=MODEL_TYPE,
        validate_files=True,
    )
    train_source, validation_source = source_dataset.train_test_split(
        validation_fraction=0.1,
        seed=42,
        group_by="session_id",
    )
    print(len(train_source), len(validation_source))

## 4. Build one training batch

In [ ]:
active_model = None
active_trainer = None

if RUN_TRAINING:
    from voicehub import AutoModelForTextToSpeech
    from voicehub.training.trainer import Trainer

    active_model = AutoModelForTextToSpeech.from_pretrained(
        BASE_MODEL,
        model_type=MODEL_TYPE,
        device=DEVICE,
        lazy_load=True,
    )
    active_model.validate_training_support()
    train_dataset = active_model.create_training_dataset(train_source)
    validation_dataset = active_model.create_training_dataset(validation_source)
    preview_batch = train_dataset.collate_fn([train_dataset[0]])
    print({name: getattr(value, "shape", type(value).__name__) for name, value in preview_batch.items()})
    active_trainer = Trainer(
        model=active_model,
        args=smoke_arguments,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
    )

## 5. Run one step and export

In [ ]:
final_artifact = OUTPUT_DIR / "final"
if active_trainer is not None:
    train_output = active_trainer.train(resume_from_checkpoint=False)
    if not isinstance(train_output.training_loss, float):
        raise RuntimeError("Training did not return a scalar loss")
    active_trainer.save_model(final_artifact)
    print(train_output, final_artifact)

## 6. Reload in a fresh runtime

Use the same prompt and seed as the pre-training baseline. A trainer checkpoint is for exact resume; the final portable artifact is for inference.

In [ ]:
if RUN_RELOAD:
    from voicehub import AutoModelForTextToSpeech, TTSGenerationConfig

    if not final_artifact.is_dir():
        raise FileNotFoundError(final_artifact)
    reloaded_model = AutoModelForTextToSpeech.from_pretrained(final_artifact, device=DEVICE)
    reloaded_output = reloaded_model.generate(
        "[S1] This is a post training reload check using the same evaluation prompt.",
        generation_config=TTSGenerationConfig(seed=42, output_file=OUTPUT_DIR / "reloaded.wav"),
    )
    print(reloaded_output.file_path)

## Next

Different model families require different fields and objectives. Before changing `MODEL_TYPE`, read the [training matrix](https://kadirnar.github.io/voicehub/models/training-support/), [training guide](https://kadirnar.github.io/voicehub/guides/training/), and [data guide](https://kadirnar.github.io/voicehub/guides/data-preparation/).